# Epigraphische Inschriften als Daten für die Analyse und Rekonstruktion des römischen Handels.
# 1. Korpuserstellung aus Inschriftendatensätze der Epigraphischen Datenbanken EDCS und EDH

## Datensätze
Die Inschriftendaten der Datenbank EDCS und EDH wurden im Projekt "Social Dynamics in the Ancient Mediterrean/SDAM" an der Universität Aarhus extrahiert. Die Datensätze sind über die folgenden Links der GitHub Repositories und auf Zenodo verfügbar: \
**EDCS:**\
DATASET 2022: Heřmánková, Petra. (2022). EDCS_text_cleaned_2022_09_12 (v2.0) [Data set] Zenodo. http://doi.org/10.5281/zenodo.4888817 \
SCRIPTS 2022: Petra Heřmánková. (2022). sdam-au/EDCS_ETL: Scripts (v2.0). Zenodo. https://doi.org/10.5281/zenodo.6497148 \
siehe auch GitHub Reporsitory unter https://github.com/sdam-au/EDCS_ETL \
\
Grundlage für das Datenset der EDCS war das Tool "LatEpig" von Ballsun-Stanton B., Heřmánková P., Laurence R. LatEpig (version 2.0). GitHub. URL: https://github.com/mqAncientHistory/Lat-Epig/ oder http://doi.org/10.5281/zenodo.12036539 .\
\
**EDH:**\
DATASET 2022: Heřmánková, Petra, & Kaše, Vojtěch. (2022). EDH_text_cleaned_2022_11_03 (v2.0) [Data set] Zenodo. http://doi.org/10.5281/zenodo.7303886 \
SCRIPTS 2022: Heřmánková, Petra, & Kaše, Vojtěch. (2022). sdam-au/EDH_ETL: Scripts (v2.0). Zenodo. https://doi.org/10.5281/zenodo.7303867 \
Metadata 2022: EDH 2022 dataset metadata mit Beschreibung aller Attribute EDH_2022_dataset_metadata_SDAM.csv .\
siehe auch GitHub Repository unter https://github.com/sdam-au/EDH_ETL \
\
Die originalen Rohdaten für das Datenset der EDH stammt aus zwei Quellen:
* Epidoc XML files verfügbar über [https://edh.ub.uni-heidelberg.de/data] (inscriptions)
* Web API verfügbar über [https://edh.ub.uni-heidelberg.de/data/api] (inscriptions and geospatial data)

Der fertiggestellte Datensatz wird als EDH_text_cleaned_[timestamp].json benannt und ist auf Zenodo (s. o.) veröffentlicht. Zusätzlich kann der identische Datensatz über Sciencedata.dk SDAM_root/SDAM_data/EDH/public folder auf sciencedata.dk oder alternativ über https://sciencedata.dk/public/b6b6afdb969d378b70929e86e58ad975/ aufgerufen werden.\
Die **aktuelleste Version der EDH** Daten von 2023 in EDH_text_cleaned_2023-04-26.json verfügbar unter https://sciencedata.dk/shared/b6b6afdb969d378b70929e86e58ad975 wurde für die Analysen verwendet, um möglichst aktuelle Daten zu untersuchen.


In [1]:
import numpy as np
import pandas as pd
import re
from unidecode import unidecode

In [2]:
# relative Pfadstrukturen
raw_folderpath = "../data/raw"
pp_folderpath = "../data/preprocess"
final_folderpath = "../data/final"

In [3]:
# Spaltenstruktur EDCS-Datei betrachten
df_edcs = pd.read_json(f"{raw_folderpath}/EDCS_text_cleaned_2022-09-12.json")
print("Anzahl Spalten:", len(df_edcs.columns))
print(df_edcs.columns)

Anzahl Spalten: 27
Index(['EDCS-ID', 'publication', 'province', 'place', 'inscription',
       'inscription_conservative_cleaning',
       'inscription_interpretive_cleaning', 'inscr_type', 'status_notation',
       'inscr_process', 'clean_text_conservative',
       'clean_text_interpretive_word', 'status', 'material', 'partner_link',
       'dating_from', 'dating_to', 'date_not_before', 'date_not_after',
       'latitude', 'longitude', 'photo', 'raw_dating', 'language', 'comment',
       'extra_text', 'extra_html'],
      dtype='object')


In [4]:
# Spaltenstruktur EDH-Datei betrachten
df_edh = pd.read_json(f"{raw_folderpath}/EDH_text_cleaned_2023-04-26.json")
print("Anzahl Spalten:", len(df_edh.columns))
print(df_edh.columns)

Anzahl Spalten: 69
Index(['commentary', 'country', 'depth', 'diplomatic_text', 'findspot_ancient',
       'findspot_modern', 'height', 'id', 'language', 'last_update',
       'letter_size', 'literature', 'material', 'modern_region', 'not_after',
       'not_before', 'responsible_individual', 'transcription',
       'trismegistos_uri', 'type_of_inscription', 'type_of_monument', 'width',
       'work_status', 'findspot', 'year_of_find', 'present_location',
       'religion', 'idno_tm', 'placenames_refs', 'text_edition',
       'origdate_text', 'layout_execution', 'layout_execution_text',
       'support_objecttype', 'support_objecttype_text', 'support_material',
       'support_material_text', 'support_decoration', 'keywords_term',
       'keywords_term_text', 'people', 'province_label', 'pleiades_id',
       'Longitude', 'Latitude', 'type_of_inscription_clean',
       'type_of_inscription_certainty', 'height_cm', 'width_cm', 'depth_cm',
       'material_clean', 'type_of_monument_clean',

Spaltenstruktur zeigt heterogene Datensätze!\
27 Spalten für EDCS-Einträge\
69 Spalten für EDH-Einträge

In [5]:
# erste Spalteneinträge ausgeben lassen
df_edcs.head()

,EDCS-ID,publication,province,place,inscription,inscription_conservative_cleaning,inscription_interpretive_cleaning,inscr_type,status_notation,inscr_process,...,date_not_before,date_not_after,latitude,longitude,photo,raw_dating,language,comment,extra_text,extra_html
0,EDCS-31400030,"CIL 03, 12297",Achaia,?,Leius,Leius,Leius,{},{},{},...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,EDCS-24700151,"CIL 01, 02650 (p 1097) = IG-05-01, 00741 = ILL...",Achaia,Afesou,Δέκιος Λείβιος (Ζ)εῦξις // D(ecimi) Leivei D(e...,Δέκιος Λείβιος εῦξις D Leivei D Leivei salve,Δέκιος Λείβιος Ζεῦξις Decimi Leivei Decimi Lei...,tituli sepulcrales,"[praenomen et nomen, viri]",{},...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,EDCS-24900077,"CIL 01, 00746 (p 944) = D 00867 = ILLRP 00374 ...",Achaia,Agia Triada / Merbaka / Midea,Q(uinto) Caecilio C(ai) f(ilio) Metel(l)o / im...,Q Caecilio C f Metelo imperatori Italici quei ...,Quinto Caecilio Cai filio Metello imperatori I...,tituli honorarii,"[officium/professio, ordo senatorius, tria nom...",{},...,-68.0,-68.0,37.638113,22.805299,http://db.edcs.eu/epigr/bilder.php?bilder.php?...,-68 to -68,NaN,NaN,NaN,NaN
3,EDCS-03700724,"ZPE-108-159 = Thesprotia 00001 = AE 1993, 0140...",Achaia,Agios Athanasios / Photike,Fortissimo et Piis/simo Caesari d(omino) n(ost...,Fortissimo et Piissimo Caesari d n Gal Val P F...,Fortissimo et Piissimo Caesari domino nostro G...,tituli honorarii,"[Augusti/Augustae, ordo equester, tria nomina]",litterae erasae,...,309.0,313.0,39.451218,20.766767,http://db.edcs.eu/epigr/bilder.php?bilder.php?...,309 to 313,NaN,NaN,NaN,NaN
4,EDCS-55701593,"AE 2009, 01286a",Achaia,Agios Donatos / Photike,Cn(aeus) Atei(us),Cn Atei,Cnaeus Ateius,tituli fabricationis,"[praenomen et nomen, viri]",sigilla impressa,...,NaN,NaN,39.475976,20.506908,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
df_edh.head()

,commentary,country,depth,diplomatic_text,findspot_ancient,findspot_modern,height,id,language,last_update,...,modern_region_clean,modern_region_certainty,findspot_modern_clean,findspot_modern_certainty,findspot_clean,findspot_certainty,origdate_text_clean,clean_text_conservative,clean_text_interpretive_word,clean_text_interpretive_sentence
0,(C): 2. Hälfte 1. - Anfang 2. Jh. - AE; Ende 1...,Italy,2 cm,D M / NONIAE P F OPTATAE / ET C IVLIO ARTEMONI...,"Cumae, bei","Cuma, bei",33 cm,HD000001,L,2014-04-07,...,Campania,Certain,"Cuma, bei",Certain,NULL,NULL,71 AD - 130 AD,D M Noniae P f Optatae et C Iulio Artemoni par...,Dis Manibus Noniae Publi filiae Optatae et Cai...,Dis Manibus Noniae Publi filiae Optatae et Cai...
1,AE 1983: Breite: 35 cm.,Italy,{},C SEXTIVS PARIS / QVI VIXIT / ANNIS LXX,Roma,Roma,28 cm,HD000002,L,2014-04-07,...,Lazio,Certain,Roma,Certain,"Via Nomentana, S. Alessandro, Kirche",Certain,51 AD - 200 AD,C Sextius Paris qui vixit annis LXX,Caius Sextius Paris qui vixit annis LXX,Caius Sextius Paris qui vixit annis LXX
2,(B): [S]isenna ist falscher Kasus; folgende Er...,Spain,(12) cm,[ ]VMMIO [ ] / [ ]ISENNA[ ] / [ ] XV[ ] / [ ] / [,{},Tomares,(37) cm,HD000003,L,2006-08-31,...,Sevilla,Certain,Tomares,Certain,NULL,NULL,131 AD - 170 AD,ummio isenna Xv,Publio Mummio Publi filio Galeria Sisennae Rut...,Publio Mummio Publi filio Galeria Sisennae Rut...
3,Material: lokaler grauer Kalkstein. (B): Stylo...,Spain,18 cm,[ ]AVS[ ]LLA / M PORCI NIGRI SER / DOMINAE VEN...,Ipolcobulcula,Carcabuey,(39) cm,HD000004,L,2015-03-27,...,Córdoba,Certain,Carcabuey,Certain,NULL,NULL,151 AD - 200 AD,AVSLLA M Porci Nigri ser dominae Veneri aram p...,AVS LLA Marci Porci Nigri serva dominae Veneri...,AVS LLA Marci Porci Nigri serva dominae Veneri...
4,(B): Z. 3: C(ai) l(ibertae) Tyches.,Italy,{},[ ] L SVCCESSVS / [ ] L L IRENAEVS / [ ] C L T...,Roma,Roma,{},HD000005,L,2010-01-04,...,Lazio,Certain,Roma,Certain,Via Cupa (ehem. Vigna Nardi),Certain,1 AD - 200 AD,l Successus L l Irenaeus C l Tyches unt renti f,libertus Successus Luci libertus Irenaeus Cai ...,libertus Successus Luci libertus Irenaeus Cai ...


In [7]:
# Anzahl Einträge in EDCS-Datei
len(df_edcs)

537262

In [8]:
# Anzahl Einträge in EDH-Datei
len(df_edh)

81883

## Vorverarbeitungschritte Merging

In [9]:
# Spaltenstruktur für das mögliche Mapping beider Dateien in df_edcs und df_edh
# (Spalte neu, Spalte EDCS, Spalte EDH)
column_label_mapping = [
    ("EDCS-ID", "EDCS-ID", ""),
    ("HD-ID", "", "id"),
    ("publication", "publication", "literature"),
    ("findspot_ancient", "place", "findspot_ancient_clean"), # zuvor ohne _clean!
    ("findspot_modern", "place", "findspot_modern_clean"),
    ("province", "province", "province_label_clean"),
    ("country", "", "country"),
    ("latitude", "latitude", "Latitude"),
    ("longitude", "longitude", "Longitude"),
    ("transcription", "inscription", "transcription"),
    ("text_conservative", "clean_text_conservative", "clean_text_conservative"),
    ("text_interpretive", "clean_text_interpretive_word", "clean_text_interpretive_word"),
    ("date_raw", "raw_dating", "origdate_text"),
    ("date_not_before", "date_not_before", "not_before"),
    ("date_not_after", "date_not_after", "not_after"),
    ("dating_from", "dating_from", ""),  # nur in EDCS
    ("dating_to", "dating_to", ""),  # nur in EDCS
    ("material", "material", "material_clean"),
    ("inscr_type", "inscr_type", "type_of_inscription_clean"),
    ("language", "language", "language"),
    ("commentary", "comment", "commentary"),
]

# Harmonisieren der Spaltenstruktur
def harmonize(df, source):
    merged_df = pd.DataFrame()
    for new_colum, edcs_colum, edh_colum in column_label_mapping:
        if source == "edcs" and edcs_colum and edcs_colum in df.columns:
            merged_df[new_colum] = df[edcs_colum]
        elif source == "edh" and edh_colum and edh_colum in df.columns:
            merged_df[new_colum] = df[edh_colum]
        else:
            merged_df[new_colum] = ""
    return merged_df

df_edcs = harmonize(df_edcs, "edcs")
df_edh = harmonize(df_edh, "edh")

In [10]:
# Spalte source für Angabe der Quelle
df_edcs["source"] = "EDCS"
df_edh["source"] = "EDH"

### Strukturelle Duplikate entfernen. Einträge mit gleicher ID-Nummer entfernen

#### Analyse der Einträge mit gleicher ID

In [11]:
# Duplikate in EDCS-Datensatz feststellen
# in Spalte 'EDCS-ID'
df_edcs["EDCS-ID"].duplicated().sum()

3955

In [12]:
# Duplikate in EDH-Datensatz feststellen
# anhand Spalte 'id'
df_edh["HD-ID"].duplicated().sum()

0

Der Datensatz für die EDH Inschriften enthält in der Spalte "HD-ID" keine Duplikate/doppelte Einträge mit selber ID-Nummer. Leider weist die Ausgabe für den Datensatz der EDCS-Einträge 3955 doppelte Einträge in der Spalte "EDCS-ID" auf.

In [13]:
# Zählen der IDs, die mehr als einmal vorkommen
# IDs, die Duplikate im Datensatz aufweisen
(df_edcs["EDCS-ID"].value_counts() > 1).sum()

3955

In [14]:
# Häufigkeitsverteilung der Duplikate
# Zählen, wie oft kommt selbe ID-Nummer vor
df_edcs["EDCS-ID"].value_counts().value_counts()

count
1    529352
2      3955
Name: count, dtype: int64

Test zeigt, dass 529.352 EDCS-IDs nur einmal vorkommen.\
3.955 IDs kommen genau zweimal vor. Aber es kommt keine ID häufiger als zweimal vor.
Nach Kontrolle in EDCS-Datenbank Online sind EDCS-IDs nur einmal vergeben. Möglicherweise kam es aufgrund fehlender API-Schnittstelle für die EDCS-Datenbank bei Datensatzerstellung des Projektes SDAM zu technisch erzeugten Duplikaten.

In [15]:
# ID-Duplikate in EDCS-ID anzeigen
# bei keep = False beide Vorkommen markieren
df_edcs[df_edcs["EDCS-ID"].duplicated(keep=False)].sort_values("EDCS-ID")

,EDCS-ID,HD-ID,publication,findspot_ancient,findspot_modern,province,country,latitude,longitude,transcription,...,date_raw,date_not_before,date_not_after,dating_from,dating_to,material,inscr_type,language,commentary,source
177184,EDCS-00380700,,"ILB 00159 = AE 1994, 01279 = AE 1995, 01100",Tongeren / Tongres / Tongern / Atuatuca,Tongeren / Tongres / Tongern / Atuatuca,Belgica | Germania inferior,,50.780627,5.463917,I(ovi) O(ptimo) M(aximo) / et Genio / mun(icip...,...,NaN,NaN,NaN,NaN,NaN,NaN,tituli sacri,NaN,NaN,EDCS
97924,EDCS-00380700,,"ILB 00159 = AE 1994, 01279 = AE 1995, 01100",Tongeren / Tongres / Tongern / Atuatuca,Tongeren / Tongres / Tongern / Atuatuca,Belgica | Germania inferior,,50.780627,5.463917,I(ovi) O(ptimo) M(aximo) / et Genio / mun(icip...,...,NaN,NaN,NaN,NaN,NaN,NaN,tituli sacri,NaN,NaN,EDCS
100202,EDCS-03000683,,"ILingons 00609 = CAG-52-02, p 74 = CAG-52-01, ...",Saints-Geosmes,Saints-Geosmes,Belgica | Germania superior,,47.833734,5.327639,Marti Bell[onae 3] / ceterisq(ue) d[is deabusq...,...,151 to 250,151.0,250.0,151.0,250.0,NaN,{},NaN,NaN,EDCS
191100,EDCS-03000683,,"ILingons 00609 = CAG-52-02, p 74 = CAG-52-01, ...",Saints-Geosmes,Saints-Geosmes,Belgica | Germania superior,,47.833734,5.327639,Marti Bell[onae 3] / ceterisq(ue) d[is deabusq...,...,151 to 250,151.0,250.0,151.0,250.0,NaN,{},NaN,NaN,EDCS
100203,EDCS-03000684,,"ILingons 00610 = CAG-52-01, p 303 = AE 1996, 0...",Saints-Geosmes,Saints-Geosmes,Belgica | Germania superior,,47.833734,5.327639,Dis Man(ibus) / Aunilla / Bellin[i f(ilia)?] / [,...,71 to 130,71.0,130.0,71.0,130.0,NaN,tituli sepulcrales,NaN,NaN,EDCS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97972,EDCS-82500193,,"Thuery-2022, 00039",Bayard-sur-Marne,Bayard-sur-Marne,Belgica | Germania superior,,48.554070,5.076992,Irascor et amo,...,NaN,NaN,NaN,NaN,NaN,aes,tituli possessionis,NaN,fibula,EDCS
176777,EDCS-82500203,,"CIL 13, 10027,162 = Thuery-2022, 00049",Flavion,Flavion,Belgica | Germania inferior,,50.249631,4.713395,Mi/sce/mi,...,NaN,NaN,NaN,NaN,NaN,aes,tituli possessionis,NaN,fibula,EDCS
97517,EDCS-82500203,,"CIL 13, 10027,162 = Thuery-2022, 00049",Flavion,Flavion,Belgica | Germania inferior,,50.249631,4.713395,Mi/sce/mi,...,NaN,NaN,NaN,NaN,NaN,aes,tituli possessionis,NaN,fibula,EDCS
98177,EDCS-82500247,,"CAG-21-02, p 287",Dijon / Divio,Dijon / Divio,Belgica | Germania superior,,47.321581,5.041470,] / [h(oc) m(onumentum) s(ive)] l(ocus) h(ered...,...,1 to 300,1.0,300.0,1.0,300.0,lapis,tituli sepulcrales,NaN,NaN,EDCS


zweites Vorkommen -> 3955 \
bei keep=False beide Vorkommen markieren -> 2*3955 = 7910 Zeilen mit selber ID

In [16]:
# Kontrolle aller doppelten ID-Einträge, ob identische Spalteneinträge
# Unterschiede bei NaN-Typ, leeren Dictionaries und leeren Listen beachten - .fillna verwenden

# .dupilcated(keep=False) markiert alle Vorkommen einer doppelten ID
duplicates = df_edcs[df_edcs["EDCS-ID"].duplicated(keep=False)]
check = (
    duplicates
    # Gruppieren der gefundenen Duplikate nach der Spalte "EDCS-ID", .groupy erzeugt für jede ID-Gruppe eigenes df
    .groupby("EDCS-ID", group_keys=False)
    # Prüfen, ob alle Zeilen identische Inhalte haben
    .apply(lambda x: (
        # fehlende Werte (NaN) durch Platzhalter ersetzt, da NaN != NaN
        x.fillna("<<NA>>") # wichtig für NaN-Vergleich und {}-Einträge, leere Listen und NaN
        # Umwandeln aller Werte in Strings für Vergleich untersch. Datentypen ({}, [], Zahlen)
         .astype(str)
        # Vergleich Zeile mit erster Zeile, .iloc[0] wählt erste Zeile der ID-Gruppe
         .eq(x.fillna("<<NA>>").astype(str).iloc[0])
        # prüft über die Spalten, ob alle Spalten pro Zeile gleich sind
         .all()
        # zweites .all() prüft, ob dies für alle Zeilen der Gruppe gilt; True = alle Einträge identisch, False = mind. ein unterschiedl. Inhalt
         .all()
    ), include_groups=False) # Warnung entfernen, groupby.apply()
)
# Zählen der identischen ID-Gruppen und Unterschiede
check.value_counts()

True    3955
Name: count, dtype: int64

Output zeigt, dass alle 3955 doppelt vorkommenden EDCS-IDs identische Spalten aufweisen (nach der NaN-Normalisierung). \
Ohne NaN-Normalisierung unterschiedliche Einträge ({} und NaN, leere Liste und NaN).\
Duplikate in EDCS_text_cleaned_2022-09-12.json gesichert keine divergierende Spalteninhalte (Kein Unterschied bei Publikation, Fundort, Datierung, Material, Textspalten, Inschriftentypen,...) - nur rein technische Duplikate, die ohne Informationsverlust entfernt werden können.

#### Entfernen der strukturellen Duplikate

In [17]:
# Bereinigung der EDCS-Einträge in EDCS_text_cleaned_2022-09-12.json
df_edcs = df_edcs.drop_duplicates(subset="EDCS-ID")

In [18]:
df_edcs["EDCS-ID"].duplicated().sum()

0

In [19]:
len(df_edcs)

533307

Im ersten Teil wurden nun die Datensätze auf Dulikate überprüft.\
Dabei wurden 3955 technisch identische Duplikate identifiziert (jeweils zwei Einträge pro EDCS-ID) und entfernt. Die Duplikate wurden zuvor auf vollständige Gleichheit in allen Spalten nach der Normalisierung fehlender Werte überprüft. Dies liefert die Grundlage für weitere Vorbereitungsschritte für das Merging der Datensätze in ein Korpus.

### Vorbereitungschritte beider heterogener Datensätze für das Merging
Unterschiedliche Inhalte in gleichen Spaltenbezeichnungen.\
Spalten mit Informationen zu Provinz, Ort/Fundorte, Material, Inschriftentypen, Sprache etc.

in EDCS Literaturangaben mit = getrennt\
in EDH Angaben mit ; getrennt. Semikolon ; aber auch Trennzeichen der Spalten, das kann eventuell zu Problemen bim Merging führen.

In [20]:
# EDH ; zu = und Leerzeichen entfernen
df_edh["publication_norm"] = df_edh["publication"].str.replace(";", "=", regex=False).str.strip()

# EDCS hat = als Trennzeichen, Leerzeichen entfernen
df_edcs["publication_norm"] = df_edcs["publication"].str.strip()

df_edh["publication_norm"]

0        AE 1983, 0192. = M. Annecchino, Puteoli 4/5, 1...
1        AE 1983, 0080. (A) = A. Ferrua, RAL 36, 1981, ...
2        AE 1983, 0518. (B) = J. González, ZPE 52, 1983...
3        AE 1983, 0533. (B) = A.U. Stylow, Gerión 1, 19...
4        AE 1983, 0078. (B) = A. Ferrua, RAL 36, 1981, ...
                               ...                        
81878    AE 2016, 1161b. = C. Engels - A. Thiel, BadFuB...
81879    W. Kersten, BJ 146, 1941, 361-362= Abb. 83 (Ze...
81880    K. Böhner - P.J. Tholen - R. von Uslar, BJ 150...
81881              M. Sommer, BJ 185, 1985, 341, Nr. 29. =
81882    B. Beyer - A. Jürgens - M. Weiß, BJ 188, 1988,...
Name: publication_norm, Length: 81883, dtype: object

#### Provinzangaben normalisieren

##### Analyse der Provinzlabel

In [21]:
# EDCS-Spalteninhalte in "province" anschauen
sorted(df_edcs["province"].unique())

['Achaia',
 'Aegyptus',
 'Aemilia / Regio VIII',
 'Africa proconsularis',
 'Alpes Cottiae',
 'Alpes Graiae',
 'Alpes Maritimae',
 'Alpes Poeninae',
 'Apulia et Calabria / Regio II',
 'Aquitani(c)a',
 'Arabia',
 'Armenia',
 'Asia',
 'Baetica',
 'Barbaricum',
 'Belgica',
 'Belgica | Germania inferior',
 'Belgica | Germania superior',
 'Britannia',
 'Bruttium et Lucania / Regio III',
 'Cappadocia',
 'Cilicia',
 'Corsica',
 'Creta et Cyrenaica',
 'Cyprus',
 'Dacia',
 'Dalmatia',
 'Etruria / Regio VII',
 'Galatia',
 'Gallia Narbonensis',
 'Germania inferior',
 'Germania superior',
 'Hispania citerior',
 'Italia',
 'Latium et Campania / Regio I',
 'Liguria / Regio IX',
 'Lugudunensis',
 'Lusitania',
 'Lycia et Pamphylia',
 'Macedonia',
 'Mauretania Caesariensis',
 'Mauretania Tingitana',
 'Mesopotamia',
 'Moesia inferior',
 'Moesia inferior | Moesia superior',
 'Moesia superior',
 'Noricum',
 'Numidia',
 'Palaestina',
 'Pannonia inferior',
 'Pannonia inferior | Pannonia superior',
 'Pannonia

In [22]:
# Nach EDH_metadaten_SDAM.csv in Spalte "province_lable_clean" bereinigte Provinznamen ohne zusätzliche Zeichen wie ?
# Ausgabe aller Kategorielabel in Spalte "material" in EDCS
sorted(df_edh["province"].unique())

['Achaia',
 'Aegyptus',
 'Aemilia (Regio VIII)',
 'Africa Proconsularis',
 'Alpes Cottiae',
 'Alpes Graiae',
 'Alpes Maritimae',
 'Alpes Poeninae',
 'Apulia et Calabria (Regio II)',
 'Aquitania',
 'Arabia',
 'Armenia',
 'Asia',
 'Baetica',
 'Barbaricum',
 'Belgica',
 'Bithynia et Pontus',
 'Britannia',
 'Bruttium et Lucania (Regio III)',
 'Cappadocia',
 'Cilicia',
 'Corsica',
 'Creta',
 'Cyprus',
 'Cyrene',
 'Dacia',
 'Dalmatia',
 'Epirus',
 'Etruria (Regio VII)',
 'Galatia',
 'Germania inferior',
 'Germania superior',
 'Hispania citerior',
 'Iudaea',
 'Latium et Campania (Regio I)',
 'Liguria (Regio IX)',
 'Lugdunensis',
 'Lusitania',
 'Lycia et Pamphylia',
 'MaE',
 'Macedonia',
 'Mauretania Caesariensis',
 'Mauretania Tingitana',
 'Mesopotamia',
 'Moesia inferior',
 'Moesia superior',
 'NULL',
 'Narbonensis',
 'Noricum',
 'Numidia',
 'Pannonia inferior',
 'Pannonia superior',
 'Picenum (Regio V)',
 'Raetia',
 'Regnum Bospori',
 'Roma',
 'Samnium (Regio IV)',
 'Sardinia',
 'Sicilia, M

In [23]:
# Bereinigen der Werte in Spalte "province" in EDCS bereinigen, damit passen zu EDH

df_edcs["province_norm"] = (
    df_edcs["province"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.replace(r"\s*/\s*Regio\s*(.+)", r" (Regio \1)", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.replace("proconsularis", "Proconsularis", regex=False)
        .str.replace("Aquitani(c)a", "Aquitania", regex=False)
)

In [24]:
df_edcs["province_norm"].unique()

array(['Achaia', 'Aegyptus', 'Aemilia (Regio VIII)',
       'Africa Proconsularis', 'Alpes Cottiae', 'Alpes Graiae',
       'Alpes Maritimae', 'Alpes Poeninae',
       'Apulia et Calabria (Regio II)', 'Aquitania', 'Arabia', 'Armenia',
       'Asia', 'Baetica', 'Barbaricum', 'Belgica',
       'Belgica | Germania inferior', 'Belgica | Germania superior',
       'Britannia', 'Bruttium et Lucania (Regio III)', 'Cappadocia',
       'Cilicia', 'Corsica', 'Creta et Cyrenaica', 'Cyprus', 'Dacia',
       'Dalmatia', 'Etruria (Regio VII)', 'Galatia', 'Gallia Narbonensis',
       'Germania inferior', 'Germania superior', 'Hispania citerior',
       'Italia', 'Latium et Campania (Regio I)', 'Liguria (Regio IX)',
       'Lugudunensis', 'Lusitania', 'Lycia et Pamphylia', 'Macedonia',
       'Mauretania Caesariensis', 'Mauretania Tingitana', 'Mesopotamia',
       'Moesia inferior', 'Moesia inferior | Moesia superior',
       'Moesia superior', 'Noricum', 'Numidia', 'Palaestina',
       'Pannonia infe

In [25]:
# In EDH Spalte "province_label_clean" nochmal bereinigen und neue Spalte mit selben Titel wie in EDCS
df_edh["province_norm"] = (
    df_edh["province"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
)

In [26]:
# Vergleich aller Provinzen aus beiden Datensätzen
# Provinzen, die nur in der jeweiligen Datei vorkommen.

edcs_provinces = df_edcs["province_norm"].unique()
edh_provinces  = df_edh["province_norm"].unique()

only_in_edcs = set(edcs_provinces) - set(edh_provinces)
only_in_edh  = set(edh_provinces)  - set(edcs_provinces)

print("Nur EDCS:", sorted(only_in_edcs))
print("Nur EDH:", sorted(only_in_edh))

Nur EDCS: ['Belgica | Germania inferior', 'Belgica | Germania superior', 'Creta et Cyrenaica', 'Gallia Narbonensis', 'Italia', 'Lugudunensis', 'Moesia inferior | Moesia superior', 'Palaestina', 'Pannonia inferior | Pannonia superior', 'Pontus et Bithynia', 'Provincia incerta', 'Sicilia']
Nur EDH: ['Bithynia et Pontus', 'Creta', 'Cyrene', 'Epirus', 'Iudaea', 'Lugdunensis', 'MaE', 'NULL', 'Narbonensis', 'Sicilia, Melita', 'TuU', 'unknown']


In [27]:
# Anzahl Inschriften mit "|", also zwei Provinzlabel
df_edcs["province_norm"].str.contains("|", regex=False).sum()

3955

In [28]:
# Prüfen, ob jede ID nur einen Eintrag in Province hat und keine weiteren Duplikate vorliegen.
df_edcs.groupby("EDCS-ID")["province_norm"].nunique().value_counts()

province_norm
1    533307
Name: count, dtype: int64

Jede einzelnde EDCS-ID hat nur einen Eintrag in province_clean. Es gibt keine ID mit unterschiedlichen Provinzangaben, die Anzahl der ID stimmt mit Anzahl der Provinzeinträge.\
Nun Label der Provinzen normalisieren für das Merging.

In [29]:
df_edh[df_edh["province_norm"] == "MaE"]["country"].value_counts()

country
Greece    6
Name: count, dtype: int64

In [30]:
df_edh[df_edh["province_norm"] == "TuU"]["country"].value_counts()

country
Italy    21
Name: count, dtype: int64

Die EDH-Provinzlabel 'MaE' und 'TuU' wurden anhand ihrer Fundort- und Landangabe den jeweiligen EDCS-Provinzlabel 'Macedonia' und 'Italia' zugeordnet. Ebenso wurden inhaltlich gleiche Provinzlabel vor dem Merging der Datensätze miteinander gemappt.

##### Normalisieren der Provinzlabel

In [31]:
# Spaltenlabel der Provinzen anpassen für Merging
# EDH-Label zu EDCS-Label mappen, da hier allgemeine Provinzlabel und EDH zu stark differenziert ist (Bsp: Creta, Cyrene)
# EDH : EDCS,
province_mapping= {
    # Subprovinzen zu allg. Label Provinz mappen, dabei können spezifische Subprovinzinformationen nicht mehr spezifisch betrachtet werden.
    'Creta' : 'Creta et Cyrenaica',
    'Cyrene' : 'Creta et Cyrenaica',
    'Narbonensis' : 'Gallia Narbonensis',
    'Epirus' : 'Achaia', # spätantike Provinz im heutigen Griechenland/Albanien
    'MaE' : 'Macedonia', # steht wohl für Macedonia/Epirus, in EDCS Macedonia
    'NULL' : 'Provincia incerta',
    'Iudaea': 'Palaestina',
    'Bithynia et Pontus' : 'Pontus et Bithynia',
    'Sicilia, Melita' : 'Sicilia',
    'TuU': 'Italia', # steht wohl für Tuscia/Umbria, in EDCS Italia
    'unknown' : 'Provincia incerta',
    "Lugudunensis": "Lugdunensis"
    }


In [32]:
# Neue Spalte province_norm normalisieren nach Dictionary der Label
df_edh["province_norm"] = df_edh["province_norm"].replace(province_mapping)
# Schreibfehler EDCS 'Lugudunensis', in EDH 'Lugdunensis' korrekt
df_edcs["province_norm"] = df_edcs["province_norm"].replace(province_mapping)

In [33]:
# Test/ Validierung des Labelmapping
edcs_provinces = set(df_edcs["province_norm"].unique())
edh_provinces  = set(df_edh["province_norm"].unique())

print("Nur EDCS:", sorted(edcs_provinces - edh_provinces))
print("Nur EDH:", sorted(edh_provinces - edcs_provinces))
# Nur EDH sollte nun leer und damit alle Kategorien angepasst und normalisiert sein.

Nur EDCS: ['Belgica | Germania inferior', 'Belgica | Germania superior', 'Moesia inferior | Moesia superior', 'Pannonia inferior | Pannonia superior']
Nur EDH: []


#### Fundorte bereiningungen und Normalisierung der Spalte place

##### Schreibweise normalisieren

In [34]:
# EDCS enthält in "place" sowohl modernen als auch antiken Fundort getrennt durch "/".
# Spalte "place" splitten nach EDH Spaltenstruktur in findspot_modern und _ancient
# modern für die weitere Analyse verwendet
df_edcs["findspot_modern"] = (
    df_edcs["findspot_modern"].str.split("/", n=1, expand=True)[0] # vor erstem /
    .str.strip()
    .replace("", pd.NA)
)

df_edcs["findspot_ancient"] = (
    df_edcs["findspot_ancient"].str.split("/", n=1, expand=True)[1] # nach erstem /
    .str.strip()
    .replace("", pd.NA)
)

df_edcs["findspot_modern_norm"] = df_edcs["findspot_modern"].copy()

In [35]:
df_edcs["findspot_ancient"].sort_values().unique()

array(["'Ain Mdila / Mdila / Midila", "'Ain el Bordj / Tigisis",
       "'Ain el-Hout", ..., 'el-Zaouia, Hr. / Thugga',
       'us Primigenius fecit sibi / et suis libertis libertabusq(ue) / posterisque eorum liberti/libertae;  tituli sepulcrales  cinerariumcf. Metropolitan Museum of Art New York; Accession number: 27.122.2a,b',
       None], dtype=object)

Es fällt ein Eintrag in der Spalte "findspot_ancient" auf, der ungewöhnlich ist bzw. das Encoding stimmt hier nicht: hier wird der Inschriftentext ausgegeben...'us Primigenius fecit sibi / et suis libertis libertabusq(ue) / posterisque eorum liberti/libertae;  tituli sepulcrales  cinerariumcf. Metropolitan Museum of Art New York; Accession number: 27.122.2a,b',\
Dieser Fehler muss genauer betrachtet werden und mögliche weitere fehlerhafte codierte Zeilen überprüft werden.

In [36]:
# Prüfen, wie der gesamte Eintrag mit der fehlerhaften Spalte in findspot_ancient aussieht.
df_edcs[df_edcs["findspot_ancient"].str.contains("Primigenius", na=False)]

,EDCS-ID,HD-ID,publication,findspot_ancient,findspot_modern,province,country,latitude,longitude,transcription,...,dating_from,dating_to,material,inscr_type,language,commentary,source,publication_norm,province_norm,findspot_modern_norm
365584,EDCS-53401577,,EDCS 00238 = Sinn 00458,us Primigenius fecit sibi / et suis libertis l...,?Dis Manibus M(arcus) Domiti,Provincia incerta,,NaN,NaN,NaN,...,81.0,130.0,NaN,{},NaN,NaN,EDCS,EDCS 00238 = Sinn 00458,Provincia incerta,?Dis Manibus M(arcus) Domiti


Die anderen Zeilen wurden korrekt eingelesen, jedoch ab place/findspot_ancient liegt hier ein struktureller Fehler im Datensatz der EDCS vor. Das ? in der Fundortspalte ist korrekt als Zeichen eines unbekannten Fundorts, jedoch fehlt danach ein Trennungszeichen für die fortlaufende Trennung der Inhalte in die anderen Spalten.

In [37]:
# weiterer Test, ob in Spalte "findspot_ancient" Einträge mit einer Länge über 180 Zeichen vorliegen.
df_edcs[df_edcs["findspot_ancient"].str.len().gt(165)]

,EDCS-ID,HD-ID,publication,findspot_ancient,findspot_modern,province,country,latitude,longitude,transcription,...,dating_from,dating_to,material,inscr_type,language,commentary,source,publication_norm,province_norm,findspot_modern_norm
308557,EDCS-31400039,,"CIL 03, 12316 = ILJug-03, 01270",[[Maxim[i]ano]] / [[et Aug(usto) n(ostro) Seve...,Nagajce[[DD(ominis) nn(ostris) Ga[l(erio)] V[a...,Macedonia,,NaN,NaN,Legerunt Premerstein et Vuli&cacute; || 1 DDNN...,...,306.0,307.0,NaN,{},NaN,NaN,EDCS,"CIL 03, 12316 = ILJug-03, 01270",Macedonia,Nagajce[[DD(ominis) nn(ostris) Ga[l(erio)] V[a...
365584,EDCS-53401577,,EDCS 00238 = Sinn 00458,us Primigenius fecit sibi / et suis libertis l...,?Dis Manibus M(arcus) Domiti,Provincia incerta,,NaN,NaN,NaN,...,81.0,130.0,NaN,{},NaN,NaN,EDCS,EDCS 00238 = Sinn 00458,Provincia incerta,?Dis Manibus M(arcus) Domiti


Zwei Zeilen im EDCS-Datensatz weisen strukturelle Fehler auf:\
EDCS-53401577: Fundort unbekannt (?), Inschriftentext fälschlicherweise im 'place'-Feld gespeichert.\
EDCS-31400039: Inschriftentext ohne Leerzeichen/ Trennungszeichen direkt an den Ortsnamen angehängt.\
Da es sich nur um zwei Zeilen handelt, die keine Informationen über Händler enthalten, werden keine Verbesserungen des Datensatzes in diese Richtung unternommen.\
Bei näherem Betrachten fällt zudem auf, dass keine Bereinigung des Inschriftentextes durch die Forschungsgruppe des Projekts SDAM hinzugefügt wurde. Daher liegt der Einlesefehler sicherlich bereits beim Export der Datenbankeinträge.

In [38]:
# Zusätze zu Fundorten in der EDH-Datei ausgeben
df_edh["findspot_modern"].dropna().str.extract(r",\s*(.*)$")[0].value_counts().head(60)

0
bei                                            3329
zwischen                                        610
Ostia                                           575
Karpoš                                           68
Sopot                                            57
Stari Grad                                       51
Crveni Krst                                      36
unbestimmt                                       26
Insel                                            24
Menorca                                          24
Butel                                            24
aus                                              22
Centar                                           20
Grocka                                           15
Fatih                                            13
Medijana                                         10
Barajevo                                          9
östlich von                                       7
Merkez                                            7
Vračar    

In [39]:
# Zusatz extrahieren nach Komma
df_edh["findspot_modern_zusatz"] = df_edh["findspot_modern"].str.extract(r",\s*(.*)$")[0].replace("", pd.NA)

# Fundort bereinigen, nur erster Ortsname
df_edh["findspot_modern_norm"] = df_edh["findspot_modern"].str.replace(r",.*$", "", regex=True)

In [40]:
# Suche nach Vorkommen Fundort mit Diakritika in EDCS
df_edcs[
    df_edcs["findspot_modern_norm"]
        .str.contains(r"[^\x00-\x7F]", regex=True, na=False)
]["findspot_modern_norm"].unique()

array(['Aïn Bouchekif', 'Taschìn'], dtype=object)

In [41]:
# Test Vorkommen Fundorte mit Diakritika und Umlauten
df_edh[
    df_edh["findspot_modern_norm"]
        .str.contains(r"[^\x00-\x7F]", regex=True, na=False)
]["findspot_modern_norm"].unique()

array(['Cañete la Real', 'Cañales de la Sierra', 'Alvarães', ...,
       'Ribeauvillé', 'Châtillon', 'Dumbrăveni'], dtype=object)

in EDCS-Spalte findspot_modern fehlen bei bis auf zwei Orten die Diakritika und Umlaute. (Köln z.B. nicht aufgeführt, in EDCS als Koln gelistet). Die Spalte findspot_ancient weist einige Unsicherheiten auf, bedingt durch historische Schwierigkeit der Rekonstruktion damaliger Ortslokalisation...\
Auch in der EDH-SPalte findspot_modern_norm Lokalisation spezifisch wie möglich durch Zusätze wie "bei", "zwischen".

In [42]:
# Normalisierung der Ortsnamen EDCS und EDH
# In EDH und EDCS Spalte mit modernem Ortsnamen normalisieren, Umlaute behalten;

def normalize_place_series(series):
    return (
        series
            .astype("string")
            .str.strip()
            # Anführende Transliterationen 'ayn als Apostroph (') vor arabischem Ortsname normalisieren
            .str.replace(r"^[`´‘]+", "'", regex=True)
            .replace("null", pd.NA)
    )

df_edh["findspot_modern_norm"] = normalize_place_series(df_edh["findspot_modern_norm"])
df_edcs["findspot_modern_norm"] = normalize_place_series(df_edcs["findspot_modern_norm"])

In [43]:
# ASCII Spalte der Fundorte als Hilfe für Matching
# Diakritika entfernen
def remove_diacritics(series):
    return series.astype("string").apply(lambda x: unidecode(x) if pd.notna(x) else x)

df_edcs["findspot_modern_norm"] = remove_diacritics(df_edcs["findspot_modern_norm"])
df_edh["findspot_modern_norm"] = remove_diacritics(df_edh["findspot_modern_norm"])


In [44]:
# Bereiningung
df_edcs["latitude"] = df_edcs["latitude"].map(lambda x: None if x == {} else x).astype(float)
df_edcs["longitude"] = df_edcs["longitude"].map(lambda x: None if x == {} else x).astype(float)

df_edh["latitude"] = df_edh["latitude"].map(lambda x: None if x == {} else x).astype(float)
df_edh["longitude"] = df_edh["longitude"].map(lambda x: None if x == {} else x).astype(float)

#### Inschriftentyp Bereiningung der Spalten zum Inschiftentyp

In [45]:
# SPalte inscr_type untersuchen und vergleichen für Harmonisierung
df_edcs["inscr_type"].head(40)

0                                   {}
1                   tituli sepulcrales
2                     tituli honorarii
3                     tituli honorarii
4                 tituli fabricationis
5                 tituli fabricationis
6                 tituli fabricationis
7                 tituli fabricationis
8                 tituli fabricationis
9                                   {}
10                    tituli honorarii
11                                  {}
12                                  {}
13               [leges, tituli sacri]
14                                  {}
15                                  {}
16                    tituli honorarii
17                            miliaria
18       [tituli operum, tituli sacri]
19                               leges
20            [carmina, tituli operum]
21                                  {}
22                                  {}
23                        tituli sacri
24                                  {}
25                       

In [46]:
# Bereinigung
def clean_inscr_type(value):
    # NaN, leere Zellen oder dict soll leere Liste erstellen
    if value is None or (isinstance(value, float) and np.isnan(value)) or isinstance(value, dict):
        return []

    # Wenn String "{}" eine leere Liste
    if isinstance(value, str) and value.strip() == "{}":
        return []

    # Wenn bereits Liste
    if isinstance(value, list):
        return [str(v).strip().lower() for v in value if v]

    # Alles andere als String behandeln, z.B. mit Komma getrennt
    value = str(value)
    value = re.sub(r"[\[\]'\"]", "", value)
    return [v.strip().lower() for v in value.split(",") if v.strip()]

# Spalte bereinigen
df_edcs["inscr_type_norm"] = df_edcs["inscr_type"].apply(clean_inscr_type)

In [47]:
all_types_list = sorted(
    df_edcs["inscr_type_norm"].explode()
    .dropna()
    .astype(str)
    .unique()
)
print(f"EDCS-Inschriftentypen:", all_types_list)

EDCS-Inschriftentypen: ['carmina', 'defixiones', 'diplomata militaria', 'inscriptiones christianae', 'leges', 'miliaria', 'reges', 'senatus consulta', 'signacula', 'termini', 'tituli fabricationis', 'tituli honorarii', 'tituli operum', 'tituli possessionis', 'tituli sacri', 'tituli sepulcrales']


In [48]:
# Nun Spalte "type_of_inscription_clean" in EDH

all_types_list = sorted(
    df_edh["inscr_type"].explode()
    .dropna()
    .astype(str)
    .unique()
)
print(f"EDH-Inschriftentypen:", all_types_list)

EDH-Inschriftentypen: ['NULL', 'acclamation', 'adnuntiatio', 'assignation inscription', 'boundary inscription', 'building/dedicatory inscription', 'calendar', 'defixio', 'elogium', 'epitaph', 'honorific inscription', 'identification inscription', 'label', 'letter', 'list', 'mile-/leaguestone', 'military diploma', 'owner/artist inscription', 'prayer', 'private legal inscription', 'public legal inscription', 'seat inscription', 'votive inscription']


In [49]:
# Manuelle Zuordnung der Inschriftentypen für Anpassung
# Mapping EDH-Typen zu EDCS-Typen (s. Überblick_zu_Inschriftentypen_material_EDCS_EDH.csv)
# EDH : EDCS,
inscr_types_mapping = {
    "acclamation": "tituli honorarii",
    "adnuntiatio": "tituli honorarii",
    "assignation inscription": "tituli operum",
    "boundary inscription": "termini",
    "building/dedicatory inscription":"tituli operum",
    "calendar": "leges",
    "defixio": "defixiones",
    "elogium": "tituli honorarii",
    "epitaph": "tituli sepulcrales",
    "honorific inscription": "tituli honorarii",
    "identification inscription": "tituli possessionis",
    "label": "tituli possessionis",
    "letter": "leges",
    "list": "leges",
    "mile-/leaguestone": "miliaria",
    "military diploma": "diplomata militaria",
    "owner/artist inscription": "tituli possessionis",
    "prayer": "tituli sacri",
    "private legal inscription": "leges",
    "public legal inscription": "leges",
    "seat inscription": "termini",
    "votive inscription": "tituli sacri",
    "NULL": "{}"
}

Nur die Bezeichnungen sollen gemappt werden, bei mehreren Nennungen von Inschriftentypen zum jeweiligen Inschrifteneintrag soll keine Normalisierung erfolgen, um spätere Statistiken nicht unnötig zu verzerren.

In [50]:
# Bereinigung der EDH-Spalte
def clean_edh_type(value):
    if pd.isna(value) or value == "{}":
        return []

    # Wenn bereits Liste vorhanden
    if isinstance(value, list):
        return [str(v).strip().lower() for v in value if v]

    # Alles andere als String, z.B. mit Komma getrennt
    value_str = str(value)
    value_str = re.sub(r"[\[\]'\"]", "", value_str)
    return [v.strip().lower() for v in value_str.split(",") if v.strip()]

df_edh["inscr_type_norm"] = df_edh["inscr_type"].apply(clean_edh_type)

In [51]:
# Mapping der EDH_labels zu EDCS_labels
def map_edh_labels(values, mapping):
    return [mapping[val] if val in mapping else val for val in values]

df_edh["inscr_type_norm"] = df_edh["inscr_type_norm"].apply(
    lambda x: map_edh_labels(x, inscr_types_mapping)
)

In [52]:
# Test Labelmapping erfolgreich
df_edh[["inscr_type", "inscr_type_norm"]].head(10)

,inscr_type,inscr_type_norm
0,epitaph,[tituli sepulcrales]
1,epitaph,[tituli sepulcrales]
2,honorific inscription,[tituli honorarii]
3,votive inscription,[tituli sacri]
4,epitaph,[tituli sepulcrales]
5,epitaph,[tituli sepulcrales]
6,epitaph,[tituli sepulcrales]
7,epitaph,[tituli sepulcrales]
8,defixio,[defixiones]
9,epitaph,[tituli sepulcrales]


In [53]:
# Einträge mit mehr als einem Label (tituli sacri, tituli sepulcrales,..)
(df_edh["inscr_type_norm"].apply(len) > 1).sum()

0

In [54]:
df_edcs["inscr_type_norm"].head(40)

0                                   []
1                 [tituli sepulcrales]
2                   [tituli honorarii]
3                   [tituli honorarii]
4               [tituli fabricationis]
5               [tituli fabricationis]
6               [tituli fabricationis]
7               [tituli fabricationis]
8               [tituli fabricationis]
9                                   []
10                  [tituli honorarii]
11                                  []
12                                  []
13               [leges, tituli sacri]
14                                  []
15                                  []
16                  [tituli honorarii]
17                          [miliaria]
18       [tituli operum, tituli sacri]
19                             [leges]
20            [carmina, tituli operum]
21                                  []
22                                  []
23                      [tituli sacri]
24                                  []
25                      [

In [55]:
# Einträge mit mehr als einem Label (leges, tituli sacri, tituli honorarii)
(df_edcs["inscr_type_norm"].apply(len) > 1).sum()

75241

#### Material bereiningen

In [56]:
# Spalte "material" in df_edcs
all_materials_list = sorted(
    df_edcs["material"].explode()
    .dropna()
    .astype(str)
    .unique()
)
print(f"EDCS-Materialkatergorien:", all_materials_list)

EDCS-Materialkatergorien: ['aes', 'argentum', 'aurum', 'corium', 'cyprum', 'ferrum', 'gemma', 'lapis', 'lignum', 'musivum', 'opus figlinae', 'orichalcum', 'os', 'plumbum', 'rupes', 'steatitis', 'sucineus', 'tectorium', 'textum', 'vitrum']


In [57]:
df_edcs["material"].head(10)

0              NaN
1            lapis
2              NaN
3              NaN
4    opus figlinae
5    opus figlinae
6    opus figlinae
7    opus figlinae
8    opus figlinae
9            lapis
Name: material, dtype: object

In [58]:
df_edcs["material"].value_counts()

material
opus figlinae    142638
lapis             93033
aes                6073
plumbum            3981
tectorium          3754
lignum             1770
musivum             934
argentum            855
vitrum              671
aurum               488
os                  437
rupes               436
gemma               426
steatitis           268
ferrum              184
cyprum               69
corium               35
orichalcum           10
sucineus              6
textum                2
Name: count, dtype: int64

In [59]:
# Bereinigung Werte jeder Zelle in Liste
def clean_material(value):
    # NaN, {} und Strings bereinigen
    if pd.isna(value) or value == "{}":
        return [] # Liste erstellen
    if isinstance(value, list):
        return [str(v).strip().lower() for v in value if v]
     # Alles andere als String in Liste (wenn nötig mit Komma getrennt)
    val_str = str(value)
    val_str = re.sub(r"[\[\]'\"]", "", val_str)
    return [v.strip().lower() for v in val_str.split(",") if v.strip()]

# Korrektur Schreibfehler von sucineus zu sucinus (sucinus,-a,-um = aus Bernstein)
def correct_sucineus(labels):
    return ["sucinus" if x == "sucineus" else x for x in labels]

# Neue Spalte erstellen und Schreibfehler verbessern
df_edcs["material_norm"] = (df_edcs["material"].apply(clean_material).apply(correct_sucineus))

In [60]:
df_edcs["material_norm"].head(10) # .head(40)

0                 []
1            [lapis]
2                 []
3                 []
4    [opus figlinae]
5    [opus figlinae]
6    [opus figlinae]
7    [opus figlinae]
8    [opus figlinae]
9            [lapis]
Name: material_norm, dtype: object

In [61]:
# EDCS: Liste auf String reduzieren (erstes Element) oder None, wenn leer
df_edcs["material_norm"] = df_edcs["material_norm"].apply(lambda x: x[0] if x else None)

In [62]:
# EDH Spalte "material_clean" normalisieren
df_edh["material"].head(10) # .head(40)

0        Marble
1        Marble
2        Marble
3     Limestone
4          NULL
5     Limestone
6    Travertine
7        Marble
8         Metal
9          NULL
Name: material, dtype: object

In [63]:
# Spalte "material_clean" in df_edcs
all_materials_list = sorted(
    df_edh["material"].explode()
    .dropna()
    .astype(str)
    .unique()
)
print(f"EDH-Materialkatergorien:", all_materials_list)

EDH-Materialkatergorien: ['Alabaster', 'Amber', 'Andesit', 'Basalt', 'Bone', 'Clay', 'Dolomit', 'Glass', 'Gneiss', 'Granit', 'Hematit', 'Ivory', 'Jet', 'Lava', 'Leather', 'Limestone', 'Marble', 'Metal', 'NULL', 'Other', 'Peperin', 'Plaster', 'Porphyr', 'Rock', 'Sandstone', 'Slate', 'Steatit', 'Travertine', 'Tuff', 'Wood']


Label der EDH auf Englisch. \
Mapping der EDH Materiallabels zu EDCS-Labels\
in "Überblick_zu_Inschriftentypen_material_EDCS_EDH.csv"

In [64]:
# Bereinigung und mapping der Labels in Spalte "material" (EDCS) und "material_clean" (EDH)

material_mapping = {
    "alabaster": "lapis",
    "amber": "sucinus",
    "andesit": "lapis",
    "basalt": "lapis",
    "bone": "os",
    "clay": "opus figlinae",
    "dolomit": "lapis",
    "glass": "vitrum",
    "gneiss": "lapis",
    "granit": "lapis",
    "hematit": "lapis",
    "ivory": "os",
    "jet": "lapis",
    "lava": "lapis",
    "leather": "corium",
    "marble": "lapis",
    "metal": "aes",
    "other": "alii",
    "peperin": "lapis",
    "plaster": "tectorium",
    "porphyr": "lapis",
    "rock": "lapis",
    "sandstone": "lapis",
    "slate": "lapis",
    "steatit": "steatitis",
    "travertine": "lapis",
    "tuff": "lapis",
    "wood": "lignum",
    "null": None  # oder auch '{}' # NaN besser
}

# Funktion zur Normalisierung der englischen Labels
def normalize_material_edh(value):
    if pd.isna(value) or str(value).strip().lower() in ["null", ""]:
        return None
    value_lower = str(value).strip().lower()
    return material_mapping.get(value_lower)  # .get verhindert KeyError

# Für Spalte "material_clean" anwenden und neue erstellen mit Strings
df_edh["material_norm"] = df_edh["material"].apply(normalize_material_edh)

In [65]:
# Test Labelmapping erfolgreich
df_edh[["material", "material_norm"]].head(10)

,material,material_norm
0,Marble,lapis
1,Marble,lapis
2,Marble,lapis
3,Limestone,None
4,NULL,None
5,Limestone,None
6,Travertine,lapis
7,Marble,lapis
8,Metal,aes
9,NULL,None


In [66]:
df_edcs[["material", "material_norm"]].head(10)

,material,material_norm
0,NaN,None
1,lapis,lapis
2,NaN,None
3,NaN,None
4,opus figlinae,opus figlinae
5,opus figlinae,opus figlinae
6,opus figlinae,opus figlinae
7,opus figlinae,opus figlinae
8,opus figlinae,opus figlinae
9,lapis,lapis


#### Sprache der Inschriften bereinigen/normalisieren

In [67]:
# Vorbereitung für Sprachbestimmung # s. a. Documentation Python 3.12: https://docs.python.org/3.12/library/re.html

# Reguläre Ausdrücke für Sprache
GREEK_CHARS = re.compile(r"[α-ωΑ-Ω]")
LATIN_CHARS = re.compile(r"[A-Za-z]")

def normalize_language(row):
    lang = row.get("language")

    # Nur Strings prüfen, alles andere ignorieren
    if isinstance(lang, str):
        lang_upper = lang.upper()
        if lang_upper in {"L", "GR", "LGR"}:
            return lang_upper
        if lang_upper == "GL":
            return "LGR"
        if lang_upper == "G":
            return "GR"

    # Text prüfen, falls kein Label oder unbekannt
    text = row.get("text_interpetive")
    if pd.isna(text) or str(text).strip() in ["{}", ""]:
        return np.nan

    text_str = str(text)
    in_greek = bool(GREEK_CHARS.search(text_str))
    in_latin = bool(LATIN_CHARS.search(text_str))

    if in_greek and in_latin:
        return "LGR"
    elif in_greek:
        return "GR"
    elif in_latin:
        return "L"
    else:
        return np.nan

df_edcs["language_norm"] = df_edcs.apply(normalize_language, axis=1)
df_edh["language_norm"]   = df_edh.apply(normalize_language, axis=1)

In [68]:
# Test Labelmapping erfolgreich
df_edcs[["language", "language_norm"]].head(20)

,language,language_norm
0,NaN,NaN
1,NaN,NaN
2,NaN,NaN
3,NaN,NaN
4,NaN,NaN
5,NaN,NaN
6,NaN,NaN
7,NaN,NaN
8,NaN,NaN
9,NaN,NaN


#### Datierungsangaben normalisieren

In [69]:
df_edcs.columns

Index(['EDCS-ID', 'HD-ID', 'publication', 'findspot_ancient',
       'findspot_modern', 'province', 'country', 'latitude', 'longitude',
       'transcription', 'text_conservative', 'text_interpretive', 'date_raw',
       'date_not_before', 'date_not_after', 'dating_from', 'dating_to',
       'material', 'inscr_type', 'language', 'commentary', 'source',
       'publication_norm', 'province_norm', 'findspot_modern_norm',
       'inscr_type_norm', 'material_norm', 'language_norm'],
      dtype='object')

In [70]:
# Spalten mit Datierungsangaben
df_edcs[['dating_from', 'dating_to', 'date_not_before', 'date_not_after', 'date_raw']] #.dtypes # Datentyp feststellen

,dating_from,dating_to,date_not_before,date_not_after,date_raw
0,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN
2,-68.0,-68.0,-68.0,-68.0,-68 to -68
3,309.0,313.0,309.0,313.0,309 to 313
4,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...
537257,1.0,200.0,1.0,200.0,1 to 200
537258,1.0,200.0,1.0,200.0,1 to 200
537259,1.0,200.0,1.0,200.0,1 to 200
537260,-25.0,25.0,-25.0,25.0,-25 to 25


In [71]:
df_edh.columns

Index(['EDCS-ID', 'HD-ID', 'publication', 'findspot_ancient',
       'findspot_modern', 'province', 'country', 'latitude', 'longitude',
       'transcription', 'text_conservative', 'text_interpretive', 'date_raw',
       'date_not_before', 'date_not_after', 'dating_from', 'dating_to',
       'material', 'inscr_type', 'language', 'commentary', 'source',
       'publication_norm', 'province_norm', 'findspot_modern_zusatz',
       'findspot_modern_norm', 'inscr_type_norm', 'material_norm',
       'language_norm'],
      dtype='object')

In [72]:
df_edh[['date_not_after',  'date_not_before', 'date_raw']] #.dtypes # Datentyp feststellen

,date_not_after,date_not_before,date_raw
0,130,71,71 AD - 130 AD
1,200,51,51 AD - 200 AD
2,170,131,131 AD - 170 AD
3,200,151,151 AD - 200 AD
4,200,1,1 AD - 200 AD
...,...,...,...
81878,200,101,101 AD - 200 AD
81879,300,1,1 AD - 300 AD
81880,300,1,1 AD - 300 AD
81881,250,151,151 AD - 250 AD


In [73]:
# Normalisierung Datierungsangaben für Vergleichbarkeit der Einträge

# EDCS-Datierung normalisieren und neue Spalte _norm
# date_not_before und date_not_after bereits float64
df_edcs["date_not_before_norm"] = df_edcs["date_not_before"]
df_edcs["date_not_after_norm"]  = df_edcs["date_not_after"]

mask = (
    df_edcs["date_not_before_norm"].notna() & # Verknüpfung bei boolean-serien
    df_edcs["date_not_after_norm"].notna()
)

#
df_edcs["date_raw_norm"] = pd.NA
df_edcs.loc[mask, "date_raw_norm"] = (
    df_edcs.loc[mask, "date_not_before_norm"].astype(int).astype(str)
    + "-"
    + df_edcs.loc[mask, "date_not_after_norm"].astype(int).astype(str)
)

# EDH-Datierung normalisieren und neue Spalte _norm
# coerce wandelt alles, das keine Zahl ist/leere Strings in NaN. Danach numerische Spalte float64
df_edh["date_not_before_norm"] = pd.to_numeric(df_edh["date_not_before"], errors="coerce")
df_edh["date_not_after_norm"]  = pd.to_numeric(df_edh["date_not_after"], errors="coerce")

mask_edh = (
    df_edh["date_not_before_norm"].notna() & # Verknüpfung bei boolean-serien
    df_edh["date_not_after_norm"].notna()
)

# Date-raw wieder erstellen
df_edh["date_raw_norm"] = pd.NA
df_edh.loc[mask_edh, "date_raw_norm"] = (
    df_edh.loc[mask_edh, "date_not_before_norm"].astype(int).astype(str)
    + "-"
    + df_edh.loc[mask_edh, "date_not_after_norm"].astype(int).astype(str)
)

 Für das Merging haben die beiden Spalten date_not_before_norm und date_not_after_norm den gleichen Datentyp und die Zeitintervalle in date_raw-norm sind ebenso normalisiert und als String geeignet für das Matching.

### Textduplikate und Unikate finden aus beiden Dateien vor dem Merging

##### Normalisierung der Inschriftentexte

In [74]:
# Merging vorbereiten mit Suche nach Duplikaten und Unikaten zwischen EDCS und EDH-EInträgen

def text_normalize(text):
    if pd.isna(text):
        return pd.NA # besser NaN statt "" leerer String für Merging
    text = str(text).lower()
    # Sonderzeichen entfernen, aber griechische Buchstaben behalten
    # Unicode-Bereich für Griechisch α-ω, Α-Ω wird erhalten
    text = re.sub(r"[^a-z0-9α-ωΑ-Ω ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text if text else pd.NA

# Texte normalisieren in beiden Textspalten für Merging
df_edcs["text_interpretive_norm"] = df_edcs["text_interpretive"].apply(text_normalize)
df_edh["text_interpretive_norm"] = df_edh["text_interpretive"].apply(text_normalize)

df_edcs["text_conservative_norm"] = df_edcs["text_conservative"].apply(text_normalize)
df_edh["text_conservative_norm"] = df_edh["text_conservative"].apply(text_normalize)

In [75]:
# Test, ob Normalisierung in Dateien erfolgreich war
# Ausgabe leerer Zellen (NaN) und leerer Strings ("")

# gefundene NaN in der Datei
print(df_edcs["text_interpretive_norm"].isna().sum())
# gefundene leere Strings
print((df_edcs["text_interpretive_norm"] == "").sum())

# gefundene NaN in der Datei
print(df_edh["text_interpretive_norm"].isna().sum())
# gefundene leere Strings
print((df_edh["text_interpretive_norm"] == "").sum())

19553
0
27
0


-> 19553 (EDCS) und 27 (EDH) fehlende Texte wurden nun erfolgreich mit NaN als fehlende Werte markiert. Wichtig für Merging, um Falsch Positive zu vermeiden zwischen Leeren Strings und fehlenden Werten ("" und NaN)

13779 verschiedene Duplikate kommen in beiden Datenbanken vor./
 Davon sind 13.149 echte Korrespondenzen zwischen dem Datensatz der EDCS und der EDH./
 516 Keys sind keine echten Korrespondenzen, sondern Mehrfachvorkommen in einer der Datenbanken. Im Folgenden werden für den Merge die echten Duplikate aus der EDH nicht berücksichtigt. Alle anderen Einträge aus der EDH bleiben erhalten.

### Kombination beider Datensätze

In [76]:
df_merged = pd.concat([df_edcs,df_edh], ignore_index=True)

In [77]:
df_merged.columns

Index(['EDCS-ID', 'HD-ID', 'publication', 'findspot_ancient',
       'findspot_modern', 'province', 'country', 'latitude', 'longitude',
       'transcription', 'text_conservative', 'text_interpretive', 'date_raw',
       'date_not_before', 'date_not_after', 'dating_from', 'dating_to',
       'material', 'inscr_type', 'language', 'commentary', 'source',
       'publication_norm', 'province_norm', 'findspot_modern_norm',
       'inscr_type_norm', 'material_norm', 'language_norm',
       'date_not_before_norm', 'date_not_after_norm', 'date_raw_norm',
       'text_interpretive_norm', 'text_conservative_norm',
       'findspot_modern_zusatz'],
      dtype='object')

##### Koordinaten vergleichen. Gleiche Koordiaten mit selber Ortsbezeichnung

In [78]:
# Geographisches Matching.
df_merged = df_merged.sort_values(["latitude", "longitude", "source", "findspot_modern_norm"])

#print() ####### ERGEBNIS VORHER UND NACHHER ANZEIGEN LASSEN

# Gruppiern nach latitude und longitude der Datensätze und sortieren nach der Quelle
df_merged["findspot_modern_norm"] = (df_merged.groupby(["latitude", "longitude"])["findspot_modern_norm"].transform("last"))

In [79]:
# Alle Einträge bleiben erhalten
print("Anzahl der Einträge mit fehlenden Koordinaten:")
print("Latitude NaN:", df_merged["latitude"].isna().sum())
print("Longitude NaN:", df_merged["longitude"].isna().sum())

Anzahl der Einträge mit fehlenden Koordinaten:
Latitude NaN: 30841
Longitude NaN: 30839


Alte Zahlen: \
Von den insgesamt 615 190 EInträgen konnten 598 462 Einräge für die geographische Normalisierung verwendet werden. 16 728 Einträge wurden ausgeschlossen, da sie über keine gültigen numerischen Koordinaten verfügten. Damit wurden einerseits EDCS-Einträge mit fehlenden Koordinaten (NaN) und andererseits EDH-Einträge mit nicht numerischen Platzhaltern {} entfernt. Alle EDCS-Einträge mit vorhandenen Koordinaten blieben nach der Normalisierung erhalten.

#### Vergleich der Datensätze durch die Spalten Text, Material und Fundort.
Nur diese Spalten vergleichen, denn bei mehr Spalten im "compare_key" besteht die Gefahr Duplikate zu verpassen.

In [80]:
# Vergleich der Datensätze anhand objektiver Spalten Text, Material und Fundort
# mehrere Spalten für präziseren Vergleich, aber nicht zu viele, sonst Gefahr Duplikate zu verpassen.

# bereinigte Spalten als Vergleichsschlüssel nutzen
df_merged["compare_key"] = (
    df_merged["text_conservative_norm"].fillna("").str.lower().str.strip() + "|" +
    df_merged["text_interpretive_norm"].fillna("").str.lower().str.strip() + "|" +
    df_merged["material_norm"].fillna("").str.lower().str.strip() + "|" +
    df_merged["findspot_modern_norm"].fillna("").str.lower().str.strip()
)

In [81]:
print(len(df_merged))
df_merged.drop_duplicates(subset=["compare_key"], inplace=True)
print(len(df_merged))

615190
532683


Um künstliche Vervielfachung aber durch die mehrfachen Vorkommen der compare_keys Korrespondenzen zu vermeiden, werden nur eindeutig indentifizierbare 1:1 Korrespondenzen zwischen EDCS und EDH zusammengeführt.\
Das interne Mehrfachvorkommen innerhalb einzelner Datenbanken werden nicht bereinigt, da sie eigenständige Objekte sind (z. B. gleicher Wortlaut und Material sowie Fundort bei Grabinschriften für verschiedene Personen).

#### ID für jede eindeutige Inschrift festlegen

In [82]:
# Inschriften-ID vergeben für spätere Analyse
# An Position 0 in der Datei stellen
# .zfill() füllt 6
df_merged.insert(0, "Inscr-ID", ["Inscr_" + str(i).zfill(6) for i in range(len(df_merged))]
)

### Speichern des erstellten Korpus

In [83]:
df_merged.columns

Index(['Inscr-ID', 'EDCS-ID', 'HD-ID', 'publication', 'findspot_ancient',
       'findspot_modern', 'province', 'country', 'latitude', 'longitude',
       'transcription', 'text_conservative', 'text_interpretive', 'date_raw',
       'date_not_before', 'date_not_after', 'dating_from', 'dating_to',
       'material', 'inscr_type', 'language', 'commentary', 'source',
       'publication_norm', 'province_norm', 'findspot_modern_norm',
       'inscr_type_norm', 'material_norm', 'language_norm',
       'date_not_before_norm', 'date_not_after_norm', 'date_raw_norm',
       'text_interpretive_norm', 'text_conservative_norm',
       'findspot_modern_zusatz', 'compare_key'],
      dtype='object')

In [84]:
# Speichern des erstellten Korpus

df_merged = df_merged[['Inscr-ID', 'EDCS-ID', 'HD-ID', 'source', 'publication_norm', 'province_norm', 'findspot_ancient', 'findspot_modern_norm', 'latitude', 'longitude',
       'inscr_type_norm', 'material_norm', 'language_norm', 'date_not_before_norm', 'date_not_after_norm', 'date_raw_norm', 'text_interpretive_norm', 'text_conservative_norm']]

df_merged = df_merged.rename(columns={
    'publication_norm': 'publication',
    'province_norm': 'province',
    'findspot_modern_norm': 'findspot_modern',
    'inscr_type_norm':'inscr_type',
    'material_norm': 'material',
    'language_norm': 'language',
    'date_not_before_norm': 'date_not_before',
    'date_not_after_norm': 'date_not_after',
    'date_raw_norm': 'date_raw',
    'text_interpretive_norm': 'text_interpretive',
    'text_conservative_norm': 'text_conservative'
})

df_merged.to_csv(f"{final_folderpath}/EDCS_EDH_mergedKorpus.csv", index=False, sep=";", encoding="utf-8-sig")

print("EDCS-Einträge:", (df_merged["source"] == "EDCS").sum())
print("EDH-Einträge:", (df_merged["source"] == "EDH").sum())
print("Einträge in mergedKorpus:", len(df_merged))

EDCS-Einträge: 454653
EDH-Einträge: 78030
Einträge in mergedKorpus: 532683


Nun kann das Korpus, das nun alle Inschriften der zur Verfügung stehenden Datenbanken-Dateien der EDCS und EDH enthält, nach Suchbegriffen der Händler und Transporteure in der römischen Antike durchsucht werden. Hierzu wird das nächste Jupyter Notebook 2.Analyse_Inschirften_Handel durchgeführt...